# Barren plateaus, and the road to QML
**Quantum Computing · Week 7 (Variational quantum circuits) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qc/w7d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Bigger circuits should be more powerful, but there's a catch: for many of them the cost landscape
becomes almost perfectly flat. The slopes shrink **exponentially** with the number of qubits, and no
optimizer can find its way. This is the **barren plateau** problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def circuit(theta, n, layers):
    """Layers of RY on every qubit and a chain of CZ gates, then a final RY layer."""
    qc = QuantumCircuit(n)
    k = 0
    for _ in range(layers):
        for q in range(n):
            qc.ry(theta[k], q)
            k += 1
        for q in range(n - 1):
            qc.cz(q, q + 1)
    for q in range(n):
        qc.ry(theta[k], q)
        k += 1
    return qc

def costs(theta, n, layers):
    """Global cost: 1 - P(all qubits 0). Local cost: 1 - the average over qubits of P(that qubit is 0)."""
    p = Statevector(circuit(theta, n, layers)).probabilities()
    index = np.arange(2 ** n)
    global_cost = 1 - p[0]
    local_cost = 1 - np.mean([p[((index >> q) & 1) == 0].sum() for q in range(n)])
    return global_cost, local_cost

def slope_spread(n, layers, which, samples=200, seed=0):
    """Standard deviation of dC/d(theta_0) over random knob settings, by the parameter-shift rule."""
    rng = np.random.default_rng(seed)
    P = n * (layers + 1)
    slopes = []
    for _ in range(samples):
        theta = rng.uniform(0, 2 * np.pi, P)
        shift = np.zeros(P)
        shift[0] = np.pi / 2
        slopes.append((costs(theta + shift, n, layers)[which] - costs(theta - shift, n, layers)[which]) / 2)
    return np.std(slopes)

print("knobs for n = 10 qubits and 10 layers:", 10 * 11)

## How big are the slopes?
For each number of qubits, pick 200 random knob settings and look at the spread of the slope along the
first knob. Two cases: a **global** cost on a **deep** circuit (as many layers as qubits), and a
**local** cost on a **shallow** circuit (2 layers).

In [ ]:
ns = [2, 4, 6, 8, 10]
global_deep = [slope_spread(n, n, 0) for n in ns]
local_shallow = [slope_spread(n, 2, 1) for n in ns]
for n, g, l in zip(ns, global_deep, local_shallow):
    print(f"n = {n:2d}: global, deep {g:.5f}   local, shallow {l:.5f}")

plt.semilogy(ns, global_deep, "o-", color="#ff8a50", label="global cost, deep circuit")
plt.semilogy(ns, local_shallow, "o-", color="#7b5cfa", label="local cost, shallow circuit")
plt.axhline(1 / (2 * np.sqrt(1000)), color="#8a85a3", ls="--", label="wobble of a 1,000-shot estimate")
plt.xlabel("number of qubits n")
plt.ylabel("typical slope size (log scale)")
plt.legend()
plt.show()

The global, deep case falls off exponentially: by 10 qubits a typical slope is about 0.001, far below the
wobble of a 1,000-shot estimate. You would need millions of shots just to tell uphill from downhill, and
more for every extra qubit. The local, shallow case shrinks much more gently.

## What flat looks like
Slices along the first knob for five random settings of the others, at 2 and at 10 qubits.

In [ ]:
xs = np.linspace(-np.pi, np.pi, 41)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True, layout="constrained")
for ax, n in zip(axes, [2, 10]):
    rng = np.random.default_rng(1)
    for _ in range(5):
        theta = rng.uniform(0, 2 * np.pi, n * (n + 1))
        values = []
        for x in xs:
            theta[0] = x
            values.append(costs(theta, n, n)[0])
        ax.plot(xs, values, lw=2)
    ax.set_title(f"{n} qubits, global cost")
    ax.set_xlabel(r"$\theta_0$")
axes[0].set_ylabel("cost")
plt.show()

## Your turn: start close to the identity
One remedy is a careful start. Instead of uniformly random angles, start with all knobs small,
$\theta \sim \mathcal{N}(0, 0.1^2)$, so the circuit begins close to doing nothing. Measure the slope spread for the
global cost on the deep 8-qubit circuit with this start, and compare with the random start.

In [ ]:
def slope_spread_small_start(n, layers, which, samples=200, seed=0, scale=0.1):
    rng = np.random.default_rng(seed)
    P = n * (layers + 1)
    slopes = []
    for _ in range(samples):
        theta = rng.uniform(0, 2 * np.pi, P)      # TODO: small angles instead: scale * rng.standard_normal(P)
        shift = np.zeros(P)
        shift[0] = np.pi / 2
        slopes.append((costs(theta + shift, n, layers)[which] - costs(theta - shift, n, layers)[which]) / 2)
    return np.std(slopes)

print("random start:", slope_spread(8, 8, 0), "  small start:", slope_spread_small_start(8, 8, 0))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def slope_spread_small_start(n, layers, which, samples=200, seed=0, scale=0.1):
    rng = np.random.default_rng(seed)
    P = n * (layers + 1)
    slopes = []
    for _ in range(samples):
        theta = scale * rng.standard_normal(P)    # close to the identity circuit
        shift = np.zeros(P)
        shift[0] = np.pi / 2
        slopes.append((costs(theta + shift, n, layers)[which] - costs(theta - shift, n, layers)[which]) / 2)
    return np.std(slopes)

print(f"random start: slope spread {slope_spread(8, 8, 0):.5f}")
print(f"small start:  slope spread {slope_spread_small_start(8, 8, 0):.5f}")
print("Near the identity the circuit isn't scrambled yet, so the landscape still has slopes to follow.")
print("It helps training get going; it doesn't make every problem easy.")